# Embedding

Mit dem Embedding wandeln wir die Chunks in Vektoren um und speichern die Daten im Chunk.

- Model: deepset/gbert-large
- Input: products_chunked.jsonl
- Output: products_embedded.jsonl

Die Texte und die Specs müssen getrennt verarbeitet werden bzw. in zwei Arrays abgelegt werden, die dann per Index wieder zusammengeführt werden. Da die Eingangsdaten bereits als Chunks strukturiert sind, genügt es die zu vektorisieren Texte zu entnehmen und die Embeddings danach wieder hinzuzufügen.

Es werden alle Daten an die DB übergeben und in 16er-Schritten encodet. Progressbar ist for fun, Normalisieren ist Standard bei ChromaDB, glaube ich. Da wir die Daten nur für die Ähnlichkeitssuche benötigen ist die Länge und die darin enthaltene semantische Bedeutung nicht relevant.

In [9]:
import json
import random
import numpy as np

from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('deepset/gbert-large')

No sentence-transformers model found with name deepset/gbert-large. Creating a new one with mean pooling.


## Daten vorbereiten

In [10]:
products_chunked = []

with open('../data/processed/products_chunked.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        products_chunked.append(json.loads(line))

# Text only
texts = [chunk['document'] for chunk in products_chunked]

# print(texts)

## Embedding

In [11]:
embeddings = model.encode(
    texts,
    batch_size = 16,
    show_progress_bar = True,
    normalize_embeddings = True,
    convert_to_numpy = True
)

Batches:   0%|          | 0/90 [00:00<?, ?it/s]

Batches: 100%|██████████| 90/90 [06:29<00:00,  4.33s/it]


## Zusammenfassen

In [12]:
for chunk, embedding in zip (products_chunked, embeddings):
    chunk['embedding'] = embedding.tolist()

with open('../data/processed/products_embedded.jsonl', 'w', encoding='utf-8') as f:
    for chunk in products_chunked:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

## Evaluieren der Embeddings

In [13]:
# Längenvergleich
assert len(embeddings) == len(products_chunked)

# Stichproben
sample_idx = random.sample(range(len(embeddings)), int(len(embeddings) * 0.01))
#sample_idx = [0]

for idx in sample_idx:
    text = products_chunked[idx]['document']
    embd = embeddings[idx]
    norm = np.linalg.norm(embd)
    test = model.encode([text], normalize_embeddings=True)[0]
    similarity = np.dot(embd, test)

    print(f"Index: {idx}")
    print(f"Text: {text[:60]}...")
    print(f"Shape: {embd}, Norm: {norm:.4f}")
    print(f"Similarity: {similarity:.8f}")

assert not np.any(np.isnan(embeddings))
assert not np.any(np.isinf(embeddings))

print(f"Shape: {embeddings.shape}")
print(f"Dtype: {embeddings.dtype}")

Index: 619
Text: Kirsch FROSTER BL-530 PRO-ACTIVE: Schubfachmaß 56 x 39 x 10 ...
Shape: [ 1.3334313e-03 -4.8443838e-03  1.0633047e-03 ... -3.2768804e-03
  7.7876917e-05 -1.8769765e-02], Norm: 1.0000
Similarity: 1.00000024
Index: 1067
Text: Der Innenraum des Kirsch LABO-520 besteht aus poliertem Alum...
Shape: [ 0.00119514 -0.00974036 -0.00237351 ...  0.00079454 -0.00837511
 -0.01572757], Norm: 1.0000
Similarity: 0.99999970
Index: 695
Text: Kirsch LABEX-340 ULTIMATE: Fahrbare Ausführung optional...
Shape: [ 0.00232569 -0.00718863 -0.00898586 ...  0.00241242 -0.00702699
 -0.00825015], Norm: 1.0000
Similarity: 1.00000012
Index: 573
Text: Kirsch FROSTER BL-730 PRO-ACTIVE: 5 Aluminium-Schubfächer mi...
Shape: [ 0.00982984 -0.00308388 -0.00266573 ... -0.00540705 -0.00511285
 -0.01789267], Norm: 1.0000
Similarity: 0.99999976
Index: 550
Text: Kirsch FROSTER BL-730 PRO-ACTIVE: Wärmeabgabe 860 Watt...
Shape: [ 1.0958580e-02  9.6679619e-03 -6.0529754e-05 ...  1.2184515e-02
  5.0952947e-03 -7.3496